In [1]:
import numpy as np
from IPython.display import Image, display
from sedona.spark import SedonaContext
import itertools
import os

In [2]:
%%capture
bucket_name = os.environ.get("SEDONA_SOURCE_BUCKET", "apache-sedona-book")

config = SedonaContext.builder()

sedona = SedonaContext.create(config.getOrCreate())

sedona.sparkContext.setLogLevel("ERROR")

sc = sedona.sparkContext
sedona.sparkContext.setCheckpointDir("checkpoint")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/30 21:07:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/30 21:07:12 WARN UDTRegistration: Cannot register UDT for org.geotools.coverage.grid.GridCoverage2D, which is already registered.
25/11/30 21:07:12 WARN SimpleFunctionRegistry: The function rs_union_aggr replaced a previously registered function.
25/11/30 21:07:12 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.geom.Geometry, which is already registered.
25/11/30 21:07:12 WARN UDTRegistration: Cannot register UDT for org.apache.sedona.common.S2Geography.Geography, which is already registered.
25/11/30 21:07:12 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.index.SpatialIndex, which is already registered.
25/11/30 21:07:12 WARN SimpleFunctionRegistry: The function st_envelop

In [3]:
import pyspark.sql.functions as f
from pyspark.sql import Window

def normalize_column(column_name: str, wage: float) -> callable:
    def transform(df):
        windowSpec = Window.partitionBy(f.lit(1)) 
        min_value = f.coalesce(f.min(column_name).over(windowSpec), f.lit(0))

        max_value = f.coalesce(f.max(column_name).over(windowSpec), f.lit(0))


        divider = f.when(
            f.col("min_value") == f.col("max_value"), f.col("min_value")
        ).otherwise(f.col("max_value") - f.col("min_value"))

        column_expression = (f.col(column_name) - f.col("min_value")) / divider

        return df \
            .withColumn("min_value", min_value)\
            .withColumn("max_value", max_value)\
            .withColumn("divider", divider)\
            .withColumn(column_name, column_expression)\
            .withColumn(column_name, wage * f.coalesce(f.col(column_name), f.lit(0)))\
            .drop("min_value", "max_value")

    return transform

In [4]:
# reading data

In [5]:
area = "POLYGON((20.8623619079589 52.0878295898438, 20.8623619079589 52.3485488891602, 21.2123298645019 52.3485488891602, 21.2123298645019 52.0878295898438, 20.8623619079589 52.0878295898438))"
intersects = f"ST_Intersects(geometry, ST_GeomFromText('{area}'))"
rs_intersects = f"RS_Intersects(rast, ST_Transform(ST_GeomFromText('{area}'), 'epsg:4326', 'epsg:3035'))"

places = (
    sedona
      .read
      .format("geoparquet")
      .load(f"s3a://{bucket_name}/source_data/places")
      .where(intersects)
)

buildings = (
    sedona
      .read
      .format("geoparquet")
      .load(f"s3a://{bucket_name}/source_data/buildings")
      .where(intersects)
)

buildings.cache().count()
buildings.createOrReplaceTempView("buildings")

In [6]:
fire_departments = (
sedona
  .read
  .format("geoparquet")
  .load(f"s3a://{bucket_name}/source_data/places")
  .where("categories.primary = 'fire_department'")
  .where(intersects)
)

fire_departments.cache().count()
fire_departments.createOrReplaceTempView("fire_departments")

police_department = (
    sedona
      .read
      .format("geoparquet")
      .load(f"s3a://{bucket_name}/source_data/places")
      .where("categories.primary = 'police_department'")
      .where(intersects)
)

police_department.cache().count()
police_department.createOrReplaceTempView("police_department")

In [7]:
fire_hydrants = (
    sedona
      .read
      .format("geoparquet")
      .load(f"s3a://{bucket_name}/source_data/infrastructure")
      .where("class = 'fire_hydrant'")
      .where(intersects)
)

fire_hydrants.cache().count()
fire_hydrants.createOrReplaceTempView("fire_hydrants")

In [8]:
population = (
    sedona
      .read
      .format("binaryFile")
      .load(f"s3a://{bucket_name}/source_data/world_population_raster")
      .selectExpr("RS_FromGeoTiff(content) AS rast")
      .where(rs_intersects)
      .selectExpr("Explode(RS_Tile(rast, 64, 64)) AS col")
      .selectExpr("col AS rast")
)

population.cache().count()
population.createOrReplaceTempView("population")

In [9]:
fire_risk = (
    sedona
      .read
      .format("binaryFile")
      .load(f"s3a://{bucket_name}/source_data/fire_risk")
      .selectExpr("risk", "RS_FromGeoTiff(content) AS rast")
      .where(rs_intersects)
      .selectExpr("risk", "Explode(RS_Tile(rast, 64, 64)) AS col")
      .selectExpr("risk", "col AS rast")
)

fire_risk.cache().count()
fire_risk.createOrReplaceTempView("fire_risk")

In [10]:
flood = (
    sedona
      .read
      .format("binaryFile")
      .load(f"s3a://{bucket_name}/source_data/flood")
      .selectExpr("rp", "RS_FromGeoTiff(content) AS rast")
      .where(rs_intersects)
      .selectExpr("rp", "Explode(RS_Tile(rast, 64, 64)) AS col")
      .selectExpr("rp", "col AS rast")
)
flood.cache().count()
flood.createOrReplaceTempView("flood")

In [11]:
flood.selectExpr("RS_MetaData(rast) AS stats").select("stats.*").show()

+----------+----------+---------+----------+------+------+-----+-----+----+-------------------+---------+----------+
|upperLeftX|upperLeftY|gridWidth|gridHeight|scaleX|scaleY|skewX|skewY|srid|numSampleDimensions|tileWidth|tileHeight|
+----------+----------+---------+----------+------+------+-----+-----+----+-------------------+---------+----------+
| 5014900.0| 3392600.0|       64|        64| 100.0|-100.0|  0.0|  0.0|3035|                  1|       64|        64|
| 5021300.0| 3392600.0|       64|        64| 100.0|-100.0|  0.0|  0.0|3035|                  1|       64|        64|
| 5027700.0| 3392600.0|       64|        64| 100.0|-100.0|  0.0|  0.0|3035|                  1|       64|        64|
| 5034100.0| 3392600.0|       64|        64| 100.0|-100.0|  0.0|  0.0|3035|                  1|       64|        64|
| 5040500.0| 3392600.0|       64|        64| 100.0|-100.0|  0.0|  0.0|3035|                  1|       64|        64|
| 5046900.0| 3392600.0|       64|        64| 100.0|-100.0|  0.0|

## Population Data

In [12]:
# it's simplification, each building can get population of the same cell, that's why the sum is much greater than the population of the city
# for our index it's ok as we look for densier areas
sedona.sql(
"""
SELECT 
    b.id,
    ST_Buffer(ST_Intersection(b.geometry, RS_Envelope(rast)), -0.00001) AS geometry,
    rast
FROM buildings AS b
JOIN population AS p ON RS_Intersects(geometry, rast)

"""
).createOrReplaceTempView("population_data")

In [13]:
sedona.sql(
"""
SELECT 
    id,
    RS_ZonalStats(rast, geometry, 1, "sum", TRUE, FALSE) AS population
FROM population_data
"""
).createOrReplaceTempView("building_population")

In [14]:
sedona.sql("select * from building_population").selectExpr("CAST(sum(population) AS double)").show()

[Stage 41:=========================================>              (26 + 9) / 35]

+-------------------------------+
|CAST(sum(population) AS DOUBLE)|
+-------------------------------+
|           4.3260531440319824E8|
+-------------------------------+



## flood risk

In [15]:
sedona.sql(
"""
SELECT 
    b.id,
    ST_Buffer(ST_Intersection(
        ST_Transform(b.geometry, 'epsg:4326', 'epsg:3035'),
        RS_Envelope(rast)
    ), -0.00001) AS geom,
    rast,
    rp
FROM buildings AS b
JOIN flood AS p ON RS_Intersects(geometry, rast)

"""
).createOrReplaceTempView("flood_data")

In [16]:
sedona.sql(
    """
    WITH flood_risk AS (
        SELECT 
            id,
            rp,
            Min(RS_ZonalStats(rast, geom, 1, "min", TRUE)) AS min_value,
            Max(RS_ZonalStats(rast, geom, 1, "max", TRUE)) AS max_value
        FROM flood_data
        GROUP BY id, rp
        Having min_value <> 'NaN' AND max_value <> 'NaN'
    )
    SELECT * FROM flood_risk
    PIVOT (
        FIRST(min_value) AS min,
        FIRST(max_value) AS max
        FOR rp IN (
            '10' AS flood_10,
            '20' AS flood_20
        )
    )

    """
).createOrReplaceTempView("flood_stats")

In [17]:
sedona.sql("select * from flood_stats").show()

[Stage 50:======================================================> (32 + 1) / 33]

+--------------------+-------------------+-------------------+-------------------+-------------------+
|                  id|       flood_10_min|       flood_10_max|       flood_20_min|       flood_20_max|
+--------------------+-------------------+-------------------+-------------------+-------------------+
|tmp_7731313033363...| 1.7940000295639038| 1.7940000295639038|  2.378000020980835|  2.378000020980835|
|tmp_7739363232333...| 1.7669999599456787| 1.7669999599456787| 2.5390000343322754| 2.5390000343322754|
|tmp_7739363233363...|   1.74399995803833|   1.74399995803833| 2.4820001125335693| 2.4820001125335693|
|tmp_7739363233363...|   1.74399995803833|   1.74399995803833| 2.4820001125335693| 2.4820001125335693|
|tmp_7739363233343...| 1.7239999771118164| 1.7239999771118164| 2.4040000438690186| 2.4040000438690186|
|tmp_7739303134303...|               NULL|               NULL|0.20499999821186066|0.20499999821186066|
|tmp_7739363430333...| 1.1640000343322754| 1.1640000343322754| 1.64600002

## fire risk

In [18]:
sedona.sql(
"""
SELECT 
    b.id,
    ST_Buffer(ST_Intersection(
        ST_Transform(b.geometry, 'epsg:4326', 'epsg:3857'),
        RS_Envelope(rast)
    ), -0.00001) AS geom,
    rast,
    risk
FROM buildings AS b
JOIN fire_risk AS p ON RS_Intersects(geometry, rast)

"""
).createOrReplaceTempView("fire_risk_data")

In [19]:
sedona.sql(
    """
    WITH fire_risk AS (
        SELECT 
            id,
            risk,
            Min(RS_ZonalStats(rast, geom, 1, "min", TRUE)) AS min_value,
            Max(RS_ZonalStats(rast, geom, 1, "max", TRUE)) AS max_value
        FROM fire_risk_data
        GROUP BY id, risk
        Having min_value <> 'NaN' AND max_value <> 'NaN'
    )
    SELECT * FROM fire_risk
    PIVOT (
        FIRST(min_value) AS min,
        FIRST(max_value) AS max
        FOR risk IN (
            'high' AS fire_risk_high,
            'intermediate' AS fire_risk_intermediate,
            'low' AS fire_risk_low
        )
    )

    """
).createOrReplaceTempView("fire_risk_stats")

In [20]:
## buildings nearby

In [21]:
sedona.sql(
    """
    WITH nearby_buildings AS (
       SELECT
            b1.id AS b1_id,
            b2.id AS b2_id
        FROM buildings AS b1
        JOIN buildings AS b2 ON ST_DWithin(b1.geometry, b2.geometry, 500, true) 
    )
    SELECT b1_id AS id, count(*) AS density FROM nearby_buildings
    GROUP BY b1_id

    """
).createOrReplaceTempView("building_density")

In [22]:
## closest fire station distance

In [23]:
sedona.sql(
    """
    SELECT 
        b.id,
        ST_DistanceSpheroid(b.geometry, f.geometry) AS distance
    FROM buildings AS b
    JOIN fire_departments AS f ON ST_KNN(b.geometry, f.geometry, 1)
    """
).createOrReplaceTempView("closest_fire_department")

In [24]:
## closest police station distance

In [25]:
sedona.sql(
    """
    SELECT 
        b.id,
        ST_DistanceSpheroid(b.geometry, p.geometry) AS distance
    FROM buildings AS b
    JOIN police_department AS p ON ST_KNN(b.geometry, p.geometry, 1)
    """
).createOrReplaceTempView("closest_police_department")

In [26]:
## closest hydrant distance

In [27]:
sedona.sql(
    """
    SELECT 
        b.id,
        ST_DistanceSpheroid(b.geometry, f.geometry) AS distance
    FROM buildings AS b
    JOIN fire_hydrants AS f ON ST_KNN(b.geometry, f.geometry, 1)
    """
).createOrReplaceTempView("closest_fire_hydrants")

In [28]:
## index

In [29]:
wages = {
    "population": 0.05,
    
    "flood_10_min": 0.1,
    "flood_10_max": 0.12,
    "flood_20_min": 0.1,
    "flood_20_max": 0.05,
    
    "fire_risk_high_min": 0.1,
    "fire_risk_high_max": 0.2,
    "fire_risk_intermediate_min": 0.03,
    "fire_risk_intermediate_max": 0.03,
    "fire_risk_low_min": 0.01,
    "fire_risk_low_max": 0.01,
    
    "density": 0.05,
    
    "closest_fire_department_distance": 0.05,
    "closest_police_department_distance": 0.05,
    "closest_fire_hydrants_distance": 0.05,
}


In [30]:
result = (sedona.sql(
    """
    SELECT 
        b.id,
        bp.population,
        f.flood_10_min,
        f.flood_10_max,
        f.flood_20_min,
        f.flood_20_max,
        fr.fire_risk_high_min,
        fr.fire_risk_high_max,
        fr.fire_risk_intermediate_min,
        fr.fire_risk_intermediate_max,
        fr.fire_risk_low_min,
        fr.fire_risk_low_max,
        bd.density,
        cf.distance AS closest_fire_department_distance,
        cp.distance AS closest_police_department_distance,
        ch.distance AS closest_fire_hydrants_distance
    FROM buildings AS b
    LEFT JOIN building_population AS bp ON bp.id = b.id
    LEFT JOIN flood_stats AS f ON f.id = b.id
    LEFT JOIN fire_risk_stats AS fr ON fr.id = b.id
    LEFT JOIN building_density AS bd ON bd.id = b.id
    LEFT JOIN closest_fire_department AS cf ON cf.id = b.id
    LEFT JOIN closest_police_department AS cp ON cp.id = b.id
    LEFT JOIN closest_fire_hydrants AS ch ON ch.id = b.id
    """
)
.transform(normalize_column("population", wages["population"]))
.transform(normalize_column("flood_10_min", wages["flood_10_min"]))
.transform(normalize_column("flood_10_max", wages["flood_10_max"]))
.transform(normalize_column("flood_20_min", wages["flood_20_min"]))
.transform(normalize_column("flood_20_max", wages["flood_20_max"]))
.transform(normalize_column("fire_risk_high_min", wages["fire_risk_high_min"]))
.transform(normalize_column("fire_risk_high_max", wages["fire_risk_high_max"]))
.transform(normalize_column("fire_risk_intermediate_min", wages["fire_risk_intermediate_min"]))
.transform(normalize_column("fire_risk_intermediate_max", wages["fire_risk_intermediate_max"]))
.transform(normalize_column("fire_risk_low_min", wages["fire_risk_low_min"]))
.transform(normalize_column("fire_risk_low_max", wages["fire_risk_low_max"]))
.transform(normalize_column("density", wages["density"]))
.transform(normalize_column("closest_fire_department_distance", wages["closest_fire_department_distance"]))
.transform(normalize_column("closest_police_department_distance", wages["closest_police_department_distance"]))
.transform(normalize_column("closest_fire_hydrants_distance", wages["closest_fire_hydrants_distance"]))
).cache()

result.count()

146402

In [31]:
result.createOrReplaceTempView("result")

In [32]:
result.selectExpr(
    "id",
    """
    (population + flood_10_min + flood_10_max + flood_20_min + flood_20_max + fire_risk_high_min + 
    fire_risk_high_max + fire_risk_intermediate_min + fire_risk_intermediate_max + fire_risk_low_min +
    fire_risk_low_max + closest_fire_department_distance + closest_police_department_distance + closest_fire_hydrants_distance) AS index
    """
).createOrReplaceTempView("index")

In [33]:
viz_area = "POLYGON((21.003947 52.225983, 21.048335 52.225983, 21.048335 52.249204, 21.003947 52.249204, 21.003947 52.225983))"
df = sedona.sql(
    """
    SELECT 
        index.id,
        index.index,
        b.geometry
    FROM index
    JOIN buildings AS b ON b.id = index.id
    """
)

In [34]:
df.cache().count()

146402

In [ ]:
from sedona.spark import SedonaKepler
import json

with open("warsaw_viz_config.json") as f:
    config_warsaw = json.load(f)

m = SedonaKepler.create_map(df, "map", config=config_warsaw)

In [ ]:
m